In [6]:
import pandas as pd
import pickle
import importlib
import sys
import os
from syllabification import tokenize, syllabify_sentences
from markov_models import MarkovModel 
from helpers import get_word_data
import warnings
import re

# define parameters
language = "FRA"
input_type = "words"

In [ ]:
n_values = [1, 2, 3, 4]  # For bigram, trigram, and 4-gram models
markov_models = {}

for n in n_values:
    
    # Create and build the Markov model
    model = MarkovModel(n)

    if input_type == "sentences": 
        # Load the paired data
        with open(f"produced_data/{language}/sentence_pairs.pkl", "rb") as f: 
            sentence_pairs = pickle.load(f)

        # Merge all transcribed (syllabified) sentences into one list
        merged_sentences = []
        for tokenized, transcribed in sentence_pairs:
            merged_sentences.extend(transcribed)

        model.build(merged_sentences, input_type)

    elif input_type == "words": 
        if language == "FRA":
            path = "Z:/data/FRA/french_lexique/Lexique383.tsv"  
        elif language == "JPN":
            path = "Z:/data/JPN/jpn.txt"
        elif language == "CMN":
            path = "Z:/data/CMN/cmn.txt"
        elif language == "VIE":
            path = "Z:/data/VIE/vie.txt"
        else:
            raise ValueError(f"Unsupported language: {language}")

        # Load the data
        words = get_word_data(path)
        print(words[:10])  # Display the first 10 words for verification

        print(f"\nTraining {n}-gram model:")
        # Build the markov model
        model.build(words, input_type)

    # Compute the conditional entropy
    # This is the information density of the model
    info_density = model.compute_conditional_entropy()
    
    print(f"Information Density for {language} and {n}-gram model: {info_density}")

    # Update the CSV file with the computed values
    update_values_in_csv(language, info_density, n)

    # Store model for later use in code (if needed)
    markov_models[n] = model

    # Display exactly 3 example joint probabilities
    example_count = 0
    print("\nExample probabilities (p(x, y)):")

    for (prefix, suffix), p_xy in model.normalized_probs.items():
        print(f"p({prefix} -> {suffix}) = {p_xy:.4f}")
        example_count += 1
        if example_count == 3:
            break
    
    # Save the model to a file
    model.save_model(language, input_type)

Language: french_lexique
[]

Training 1-gram model:
Information Density for FRA and 1-gram model: 0.0

Example probabilities (p(x, y)):

✅ Saved 1-gram model to 'produced_data/FRA/'
Language: french_lexique
[]

Training 2-gram model:
Information Density for FRA and 2-gram model: 0.0

Example probabilities (p(x, y)):

✅ Saved 2-gram model to 'produced_data/FRA/'
Language: french_lexique
[]

Training 3-gram model:
Information Density for FRA and 3-gram model: 0.0

Example probabilities (p(x, y)):

✅ Saved 3-gram model to 'produced_data/FRA/'
Language: french_lexique
[]

Training 4-gram model:
Information Density for FRA and 4-gram model: 0.0

Example probabilities (p(x, y)):

✅ Saved 4-gram model to 'produced_data/FRA/'


In [ ]:
# prepare transcribed sentences 

if language == "FRA" and input_type == "sentences": 
    path = "Z:/data/FRA/french_sentences.txt"  # Use the mounted drive letter

    # Read each line as a sentence
    tokenized_sentences = []
    transcribed_sentences = []


    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            sentence = line.strip()
            if sentence:
                # Tokenize
                tokenized_sentence = tokenize(sentence)
                print(tokenized_sentence)
                tokenized_sentences.append(tokenized_sentence)
                
                # Syllabify
                transcribed_sentence = syllabify_sentences(tokenized_sentence, language="FRA")
                print(transcribed_sentence)
                if transcribed_sentence: 
                    transcribed_sentences.append(transcribed_sentence)

            # Optional: limit for testing
            if i >= 20:
                break


    # show the entries 
    for i, sentence in enumerate(transcribed_sentences[:20]):
        print(f"Sentence {i+1}: {sentence}")

else: 
    warnings.warn("Warning: The specified language is not available yet")


# Save to .pkl
paired_sentences = list(zip(tokenized_sentences, transcribed_sentences))

with open("produced_data/{language}/preprocessed_{input_type}.pkl", "wb") as f:
    pickle.dump(paired_sentences, f)

print(f"✅ Saved tokenized and transcribed sentences to 'produced_data/{language}'")
